# DocTamper 蒸馏训练 (Colab)

在 Colab 上运行轻量模型蒸馏训练，数据与依赖加载方式与根目录 `DTD Reproduction on DocTamper.ipynb` 一致。

## 1. 挂载 Drive 并克隆仓库

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive/')

%cd /content
if not os.path.exists('/content/DocTamper'):
    !git clone -b feat-distillation-fixpath-colab https://github.com/LeSiIence/DocTamper.git
else:
    %cd /content/DocTamper
    !git fetch origin
    !git checkout feat-distillation-fixpath-colab
    !git pull --ff-only origin feat-distillation-fixpath-colab

%cd /content/DocTamper/models

## 2. 安装依赖

在 `DocTamper/models` 目录安装（与 DTD 笔记本一致）。Colab 已带 PyTorch，仅安装缺失包与 jpegio。

In [ ]:
!pip install -q lmdb albumentations segmentation_models_pytorch timm efficientnet_pytorch tqdm
!pip install -q opencv-python-headless Pillow

# jpegio 从源码安装（与根目录 ipynb 一致）
!git clone https://github.com/dwgoon/jpegio.git
%cd jpegio
!python setup.py install -q
%cd ..

## 3. 从 Drive 拷贝数据与权重

需在 Drive 中准备：
- `DocTamperV1-TrainingSet`：LMDB 训练集目录（含 CM+SP+GE 全部篡改类型）
- `checkpoints`：教师权重等（内含 `dtd_doctamper.pth`）

首次使用 TrainingSet 时需运行 `generate_pks.py` 生成压缩记录文件。

In [ ]:
import os

TargetFolder = '/content/drive/MyDrive/TargetFolder'
ProjectRoot = '/content/DocTamper'
LMDB_NAME = 'DocTamperV1-TrainingSet'

!cp -r "{TargetFolder}/{LMDB_NAME}" "{ProjectRoot}/"
assert os.path.exists(f'{ProjectRoot}/{LMDB_NAME}'), f'{LMDB_NAME} not found in TargetFolder'

assert os.path.exists(f'{ProjectRoot}/qt_table.pk'), 'qt_table.pk not found'

# 为 TrainingSet 生成 pks 文件（如已存在则跳过）
pks_path = f'{ProjectRoot}/pks/{LMDB_NAME}_75.pk'
if not os.path.exists(pks_path):
    !cd {ProjectRoot} && python generate_pks.py --lmdb_path {LMDB_NAME} --minq 75
assert os.path.exists(pks_path), f'pks generation failed: {pks_path}'

os.makedirs(f'{ProjectRoot}/pths', exist_ok=True)
!cp -r "{TargetFolder}/checkpoints" "{ProjectRoot}/pths" 2>/dev/null || true
!cp {ProjectRoot}/pths/checkpoints/dtd_doctamper.pth {ProjectRoot}/pths/ 2>/dev/null || true
print('Data and checkpoints ready.')

## 4. 启动蒸馏训练

使用 `DocTamperV1-TrainingSet`（含 CM+SP+GE 全部篡改类型）进行训练。

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python /content/DocTamper/train_distill.py \
  --data_root /content/DocTamper \
  --lmdb_name DocTamperV1-TrainingSet \
  --teacher_pth /content/DocTamper/pths/dtd_doctamper.pth \
  --save_dir /content/DocTamper/pths \
  --save_dir_drive /content/drive/MyDrive/DocTamper_ckpts \
  --batch_size 96 \
  --num_workers 8 \
  --epochs 50
  #--resume /content/DocTamper/pths/light_dtd_distill_epoch20.pth

## 5. （可选）将 checkpoint 拷回 Drive

In [ ]:
# 训练结束后，把 DocTamper/pths 下的 light_dtd_distill_*.pth 拷回 Drive 保存
!cp /content/DocTamper/pths/light_dtd_distill_*.pth '/content/drive/MyDrive/' 2>/dev/null || echo 'No light_dtd_distill checkpoints found.'